## Importar librerías y definir rutas

In [4]:
import fitz  # PyMuPDF
import os
from pathlib import Path
import pytesseract
from PIL import Image
import io
import re

# Rutas dentro del contenedor
PDF_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/pdfs")
OUTPUT_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/txt_bruto")

# Crear carpeta de salida si no existe
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Listar todos los PDFs
pdf_files = list(PDF_FOLDER.glob("*.pdf"))
print(f"Se encontraron {len(pdf_files)} archivos PDF:")
for pdf in pdf_files:
    print(f"  - {pdf.name}")

Se encontraron 49 archivos PDF:
  - DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.pdf
  - ESTATUTO 2024. 04-09-2024.pdf
  - Guía para la organización y orientación del legajo para la docencia ordinaria.pdf
  - Modelo de índice del contenido - Legajo.pdf
  - MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021.pdf
  - Politica Institucional de Inclusión y diversidad cultural v1.pdf
  - Politica Institucional de trabajo digno y protección de la persona v.1.pdf
  - Politica-ambiental.pdf
  - REGLAMENTO ADMISION 2025.v7.pdf
  - REGLAMENTO BECAS 2021 ACTUALIZADO.pdf
  - REGLAMENTO CODIGO ETICA INVESTIGACION 2021.pdf
  - Reglamento Comite Electoral Universitario 2016 OK.pdf
  - Reglamento de duplicado de diplomas.pdf
  - REGLAMENTO DE ESTUDIOS POSGRADO 2025.pdf
  - REGLAMENTO DE ESTUDIOS V5_2025.pdf
  - Reglamento de Jefes de prácticas apoyo y profesional.pdf
  - Reglamento de promocion de recreación y del deporte.pdf
  - Reglamento de Publicaciones y Fondo Editorial.pdf


## Función para extraer texto de un PDF

In [5]:
def extraer_texto_hibrido(pdf_path, umbral_palabras_pagina=15, dpi=250):
    """
    Extrae texto de un PDF página por página.
    Si una página tiene menos de umbral_palabras_pagina palabras reales (ignorando números),
    se aplica OCR sobre esa página renderizada como imagen.
    """
    texto_completo = []
    doc = fitz.open(pdf_path)
    
    for num_pag, pagina in enumerate(doc, start=1):
        # 1. Extraer texto digital de la página
        texto_digital = pagina.get_text()
        
        # Contar palabras significativas (ignorando líneas con solo números)
        lineas = texto_digital.splitlines()
        palabras = []
        for linea in lineas:
            linea_strip = linea.strip()
            if not linea_strip:
                continue
            if re.match(r'^\s*\d+\s*$', linea_strip):
                continue
            if re.search(r'\bPágina\s+\d+\b', linea_strip, re.IGNORECASE):
                continue
            palabras.extend(linea_strip.split())
        
        num_palabras = len(palabras)
        
        if num_palabras >= umbral_palabras_pagina:
            # La página tiene texto suficiente, usamos la versión digital
            texto_completo.append(f"[Página {num_pag}]\n{texto_digital}")
            print(f"  Pág {num_pag}: digital ({num_palabras} palabras)")
        else:
            # Aplicar OCR a la página
            try:
                pix = pagina.get_pixmap(dpi=dpi)
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                texto_ocr = pytesseract.image_to_string(img, lang='spa')
                if texto_ocr.strip():
                    texto_completo.append(f"[Página {num_pag}]\n{texto_ocr}")
                    print(f"  Pág {num_pag}: OCR ({len(texto_ocr.split())} palabras)")
                else:
                    texto_completo.append(f"[Página {num_pag}] (sin texto)")
                    print(f"  Pág {num_pag}: vacía (ni digital ni OCR)")
            except Exception as e:
                print(f"  Pág {num_pag}: error OCR -> {e}")
                texto_completo.append(f"[Página {num_pag}] (error OCR)")
    
    doc.close()
    return "\n\n".join(texto_completo)

## Procesar todos los PDFs y guardar texto crudo

In [6]:
for pdf_path in pdf_files:
    print(f"Procesando: {pdf_path.name} ...")
    texto = extraer_texto_hibrido(pdf_path, umbral_palabras_pagina=15, dpi=250)
    
    if texto and len(texto.strip()) > 0:
        txt_filename = pdf_path.stem + ".txt"
        txt_path = OUTPUT_FOLDER / txt_filename
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(texto)
        print(f"  -> Guardado en {txt_path.name} ({len(texto)} caracteres)\n")
    else:
        print(f"  -> Falló la extracción\n")

Procesando: DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.pdf ...
  Pág 1: digital (19 palabras)
  Pág 2: OCR (354 palabras)
  Pág 3: digital (348 palabras)
  Pág 4: digital (421 palabras)
  Pág 5: digital (357 palabras)
  Pág 6: digital (348 palabras)
  Pág 7: digital (349 palabras)
  -> Guardado en DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.txt (15099 caracteres)

Procesando: ESTATUTO 2024. 04-09-2024.pdf ...
  Pág 1: digital (40 palabras)
  Pág 2: OCR (342 palabras)
  Pág 3: digital (341 palabras)
  Pág 4: digital (516 palabras)
  Pág 5: digital (546 palabras)
  Pág 6: digital (470 palabras)
  Pág 7: digital (487 palabras)
  Pág 8: digital (443 palabras)
  Pág 9: digital (384 palabras)
  Pág 10: digital (467 palabras)
  Pág 11: digital (380 palabras)
  Pág 12: digital (370 palabras)
  Pág 13: digital (463 palabras)
  Pág 14: digital (428 palabras)
  Pág 15: digital (446 palabras)
  Pág 16: digital (446 palabras)
  Pág 17: digital (424 palabras)
  Pág 18: digital (443 palabras)


## Verificar tamaño de los archivos generados

In [7]:
txt_files = list(OUTPUT_FOLDER.glob("*.txt"))
print(f"Se generaron {len(txt_files)} archivos de texto:")
for txt in txt_files:
    size_kb = os.path.getsize(txt) / 1024
    print(f"  - {txt.name}: {size_kb:.1f} KB")

Se generaron 49 archivos de texto:
  - DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025.txt: 15.0 KB
  - ESTATUTO 2024. 04-09-2024.txt: 115.5 KB
  - Guía para la organización y orientación del legajo para la docencia ordinaria.txt: 41.9 KB
  - Modelo de índice del contenido - Legajo.txt: 2.6 KB
  - MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021.txt: 18.1 KB
  - Politica Institucional de Inclusión y diversidad cultural v1.txt: 3.6 KB
  - Politica Institucional de trabajo digno y protección de la persona v.1.txt: 3.6 KB
  - Politica-ambiental.txt: 2.4 KB
  - REGLAMENTO ADMISION 2025.v7.txt: 104.2 KB
  - REGLAMENTO BECAS 2021 ACTUALIZADO.txt: 73.5 KB
  - REGLAMENTO CODIGO ETICA INVESTIGACION 2021.txt: 26.9 KB
  - Reglamento Comite Electoral Universitario 2016 OK.txt: 27.5 KB
  - Reglamento de duplicado de diplomas.txt: 7.6 KB
  - REGLAMENTO DE ESTUDIOS POSGRADO 2025.txt: 307.1 KB
  - REGLAMENTO DE ESTUDIOS V5_2025.txt: 365.0 KB
  - Reglamento de Jefes de prácticas a

In [8]:
import fitz  # PyMuPDF

# Ruta a un PDF que sepas que tiene tablas (elige uno)
pdf_path = "/home/jupyteruser/work/corpus_upeu/pdfs/TUPA V6 2023 UPeU.pdf"
doc = fitz.open(pdf_path)

# Inspecciona una página que contenga tabla (ajusta el número)
pagina = doc[2]  # Página 5 (empieza en 0)
texto = pagina.get_text()
print(texto[:1500])  # primeros 1500 caracteres

 
3 
 
UNIVERSIDAD PERUANA UNIÓN 
SECRETARIA GENERAL 
TEXTO ÚNICO DE PROCEDIMIENTOS ACADÉMICO-ADMINISTRATIVOS (TUPA) 
 
Nº  
TRÁMITE  
REQUISITOS  
DERECHO DE  
TRÁMITE  
PERIODO O 
ADMISIÓN  
PROCESO  
/ INSTANCIA* 
  
PLAZO Y/O 
ENTREGA  
ÁREA 
RESPONSABLE  
1  
ADMISIÓN 
 
 
 
(*) Denegatoria, incumplimiento de plazos.  
 
 
1.1 Derecho de Admisión Medicina Humana 
Requisitos: 
1. Copia CE o Pasaporte para extranjeros 
2. Copia legible de Partida de nacimiento. 
3. Certificado original de estudios de 
educación básica secundaria o 
educación básica alternativa o su 
equivalente para extranjeros, 
previamente revalidado en el Ministerio 
de Educación (MINEDU) del Perú. 
4. N° Voucher y fecha de pago por derecho 
de admisión. 
5. Verificar requisitos adicionales en la 
web. 
S/. 250.00 
Solo en período 
de convocatorias 
de admisión. En el 
período específico 
determinado para 
la Carrera de 
Medicina Humana  
1. Inscripción en línea. 
2. Pago en efectivo en caja UPeU o en cuenta Banc

In [9]:
import pdfplumber

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[2]
    # Extraer solo tablas
    tables = page.extract_tables()
    for i, table in enumerate(tables):
        print(f"--- Tabla {i+1} ---")
        for row in table:
            print(row)
    # También puedes extraer texto normal con page.extract_text()

--- Tabla 1 ---
['Nº', 'TRÁMITE', 'REQUISITOS', 'DERECHO DE', None, 'PROCESO', '', 'PLAZO Y/O', None]
[None, None, None, 'TRÁMITE', None, '/ INSTANCIA*', None, 'ENTREGA', None]
[None, None, None, None, None, '', None, None, None]
['1', 'ADMISIÓN', '', '', '', '(*) Denegatoria, incumplimiento de plazos.', None, '', '']
['1.1', 'Derecho de Admisión Medicina Humana', 'Requisitos:\n1. Copia CE o Pasaporte para extranjeros\n2. Copia legible de Partida de nacimiento.\n3. Certificado original de estudios de\neducación básica secundaria o\neducación básica alternativa o su\nequivalente para extranjeros,\npreviamente revalidado en el Ministerio\nde Educación (MINEDU) del Perú.\n4. N° Voucher y fecha de pago por derecho\nde admisión.\n5. Verificar requisitos adicionales en la\nweb.', 'S/. 250.00', 'Solo en período\nde convocatorias\nde admisión. En el\nperíodo específico\ndeterminado para\nla Carrera de\nMedicina Humana', '1. Inscripción en línea.\n2. Pago en efectivo en caja UPeU o en cuenta Ba